In [2]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import google.generativeai as genai
from langchain_core.runnables import RunnableParallel
import torch
import os

load_dotenv()

True

In [18]:

google_model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash')
groq_chat_model = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

prompt_1 = PromptTemplate(template='Generate a summary on this {Topic}', input_variables = ['Topic'])


prompt_2 = PromptTemplate(template='Generate 5 quiz Questions in this {Topic}', input_variables = ['Topic'])

parser = StrOutputParser()

parallel_chain = RunnableParallel({'summary':prompt_1 | groq_chat_model | parser, 'quiz':prompt_2 | groq_chat_model | parser })

# prompt_3 = PromptTemplate(template='To merge the following {summary} and {quiz}', input_variables = ['summary','quiz'])

prompt_3 = PromptTemplate(
    template='''Combine the following summary and quiz questions. each in 50 words.

Summary:
{summary}

Quiz Questions:
{quiz}

Return both sections clearly:
1. Summary
2. Quiz Questions''',
    input_variables=['summary', 'quiz']
)

chain_3 = prompt_3 | groq_chat_model | parser

final_chain = parallel_chain | chain_3

result =  final_chain.invoke({'Topic': 'Generative Ai'})

print(result)





**1. Summary (50 words)**  
Generative AI creates original text, images, audio, code, and 3‑D content by learning data patterns. Core models include transformer language models, diffusion, VAEs, GANs, and NeRFs. Applications span content creation, visual arts, software, healthcare, education, gaming, and business. Benefits: speed, cost, personalization; challenges: hallucinations, bias, IP, security, compute, explainability issues.  

**2. Quiz Questions (50 words)**  
Quiz: test core concepts, models, prompting, evaluation, ethics.  
Q1: What defines a generative model? A:B  
Q2: Architecture used for text‑to‑image generation? A:C  
Q3: Best prompt for concise answer? A:B  
Q4: Metric for image similarity? A:C  
Q5: Not a typical generative‑AI risk? A:B  
Answers follow each question. Review and discuss now.


In [19]:
!pip install grandalf

In [20]:
final_chain.get_graph().print_ascii()

          +-----------------------------+          
          | Parallel<summary,quiz>Input |          
          +-----------------------------+          
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +----------+     
    | ChatGroq |                  | ChatGroq |     
    +----------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP

In [ ]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import google.generativeai as genai
from langchain_core.runnables import RunnableBranch , RunnableLambda
from typing import Literal
from pydantic import BaseModel , Field
import torch
import os

load_dotenv()

True

In [38]:


class Sentiment(BaseModel):
    sentiment : Literal['Positive','Negative']=Field(description = 'Generate the sentiment as either Positive or Negative')


parser = PydanticOutputParser(pydantic_object = Sentiment)
prompt_1 = PromptTemplate(template = 'Generate the Sentiment of the Following {following} in specific {format}', input_variables =['following'], partial_variables={'format':parser.get_format_instructions()})


model = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

chainS = prompt_1 | model | parser
result = chainS.invoke({
    'following': 'I really like this product'
})
result



Sentiment(sentiment='Positive')

In [42]:
prompt_2 = PromptTemplate(template = 'Generate the Positive Response for the Received {Sentiment} to the user' , input_variables=['Sentiment'] )

prompt_3 = PromptTemplate(template = 'Generate the  Response as the Sorry Message for the Received {Sentiment} to the user', input_variables=['Sentiment'])

strparser = StrOutputParser()

conditional_chain = RunnableBranch((lambda x : x.sentiment == 'Positive', prompt_2 | model | strparser), (lambda x : x.sentiment == 'Negative', prompt_3 | model | strparser), RunnableLambda(lambda x: 'Could not Extract the Sentiment')) 

final_chain = chainS | conditional_chain
final_chain.invoke({'Iphone 17 is a hype as i did not liked it all'})

final_chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
